# Data Profiling and Data Quality Check

This notebook documents the profiling and data quality work for the Paccafe sales dataset.

**Coverage**
- List all tables from the PostgreSQL database
- Extract all tables into pandas DataFrames
- Profile table shapes, column data types, and configured unique values
- Validate missing values, date fields, numeric fields, and negative values
- Save JSON reports into `mentoring_1/outputs`
- Generate dataset cleaning recommendations from the findings


# Import Library


In [30]:
from pathlib import Path
from datetime import datetime
from urllib.parse import quote_plus
import json
import tomllib

import pandas as pd
from IPython.display import display
from pandas.api.types import is_datetime64_any_dtype, is_object_dtype, is_string_dtype
from sqlalchemy import create_engine, text


# Check Kernel


In [31]:
import sys

print(sys.executable)
print(sys.version)


/Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1/.venv/bin/python
3.12.12 (main, Dec 17 2025, 21:07:08) [Clang 21.1.4 ]


## Load Config


In [32]:
def find_config_file(start_path: Path = Path.cwd()) -> Path:
    """
    Find the mentoring config file whether the notebook is run from
    the repository root or from inside `mentoring_1/notebooks`.
    """
    current_path = start_path.resolve()
    candidates: list[Path] = []

    for path in [current_path, *current_path.parents]:
        candidates.append(path / "config.toml")
        candidates.append(path / "mentoring_1" / "config.toml")

    for candidate in candidates:
        if candidate.exists():
            return candidate

    checked_paths = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "config.toml not found. Checked these paths:\n" + checked_paths
    )


CONFIG_PATH = find_config_file()
PROJECT_ROOT = CONFIG_PATH.parent

with open(CONFIG_PATH, "rb") as file:
    config = tomllib.load(file)

print("Project root:", PROJECT_ROOT)
print("Config path:", CONFIG_PATH)
print("Task file:", PROJECT_ROOT / config["paths"]["task_file"])


Project root: /Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1
Config path: /Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1/config.toml
Task file: /Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1/task_mentoring_week1.md


## Create Output Directory


In [33]:
output_dir = PROJECT_ROOT / config["paths"]["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir)


Output directory: /Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1/outputs


## Create Database Connection


In [34]:
db_config = config["database"]

username = quote_plus(db_config["username"])
password = quote_plus(db_config["password"])
host = db_config["host"]
port = db_config["port"]
database = db_config["database"]

connection_url = (
    f"postgresql+psycopg2://{username}:{password}"
    f"@{host}:{port}/{database}"
)

engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone()[0])


PostgreSQL 16.13 (Debian 16.13-1.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


# Data Profiling


## 1.1 List All Tables


In [35]:
def list_tables(engine, schema: str = "public") -> list[str]:
    """Retrieve all base tables from the selected schema."""
    query = text(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :schema
          AND table_type = 'BASE TABLE'
        ORDER BY table_name;
        """
    )

    with engine.connect() as conn:
        result = conn.execute(query, {"schema": schema})
        tables = [row[0] for row in result.fetchall()]

    return tables


In [36]:
tables = list_tables(engine, db_config["schema"])

display(pd.DataFrame({"table_name": tables}))


,table_name
0,customers
1,employees
2,inventory_tracking
3,order_details
4,orders
5,products


## 1.2 Extract Data


In [37]:
def extract_data(engine, tables: list[str], schema: str = "public") -> dict[str, pd.DataFrame]:
    """Extract all rows from the selected tables into pandas DataFrames."""
    dataframes: dict[str, pd.DataFrame] = {}

    for table_name in tables:
        query = f'SELECT * FROM "{schema}"."{table_name}"'
        dataframes[table_name] = pd.read_sql(query, engine)

    return dataframes


In [38]:
dataframes = extract_data(engine, tables, db_config["schema"])

extraction_summary = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "rows": df.shape[0],
            "columns": df.shape[1],
        }
        for table_name, df in dataframes.items()
    ]
).sort_values("table_name", ignore_index=True)

display(extraction_summary)


,table_name,rows,columns
0,customers,204,7
1,employees,103,7
2,inventory_tracking,162,6
3,order_details,3022,7
4,orders,1010,8
5,products,54,7


## 1.3 Get Row and Column Information


In [39]:
def get_table_shape(df: pd.DataFrame) -> list[int]:
    """Return `[rows, columns]` for one DataFrame."""
    return [df.shape[0], df.shape[1]]


def get_all_table_shapes(dataframes: dict[str, pd.DataFrame]) -> dict[str, list[int]]:
    """Return shape information for all extracted tables."""
    return {
        table_name: get_table_shape(df)
        for table_name, df in dataframes.items()
    }


In [40]:
table_shapes = get_all_table_shapes(dataframes)

table_shapes_frame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "rows": shape[0],
            "columns": shape[1],
        }
        for table_name, shape in table_shapes.items()
    ]
).sort_values("table_name", ignore_index=True)

display(table_shapes_frame)


,table_name,rows,columns
0,customers,204,7
1,employees,103,7
2,inventory_tracking,162,6
3,order_details,3022,7
4,orders,1010,8
5,products,54,7


## 1.4 Check Data Types of All Columns


In [41]:
def get_column_data_types(df: pd.DataFrame) -> dict[str, str]:
    """Return pandas data types for all columns in one DataFrame."""
    return {
        column_name: str(dtype)
        for column_name, dtype in df.dtypes.items()
    }


def get_all_column_data_types(dataframes: dict[str, pd.DataFrame]) -> dict[str, dict[str, str]]:
    """Return column data types for all extracted tables."""
    return {
        table_name: get_column_data_types(df)
        for table_name, df in dataframes.items()
    }


In [42]:
data_types = get_all_column_data_types(dataframes)

data_types_frame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "column_name": column_name,
            "data_type": data_type,
        }
        for table_name, columns in data_types.items()
        for column_name, data_type in columns.items()
    ]
).sort_values(["table_name", "column_name"], ignore_index=True)

display(data_types_frame)


,table_name,column_name,data_type
0,customers,created_at,datetime64[us]
1,customers,customer_id,int64
2,customers,email,str
3,customers,first_name,str
4,customers,last_name,str
5,customers,loyalty_points,int64
6,customers,phone,str
7,employees,created_at,datetime64[us]
8,employees,email,str
9,employees,employee_id,int64


## 1.5 Check Unique Values in Configured Columns


In [43]:
TABLE_NAME_ALIASES = {
    "inventory": "inventory_tracking",
}


def resolve_table_name(requested_table: str, dataframes: dict[str, pd.DataFrame]) -> str | None:
    """Map task aliases such as `inventory` to real database table names."""
    if requested_table in dataframes:
        return requested_table

    alias = TABLE_NAME_ALIASES.get(requested_table)
    if alias in dataframes:
        return alias

    return None


def get_unique_values(df: pd.DataFrame, column_name: str) -> list[str]:
    """Return sorted unique non-null, non-blank values from one column."""
    if column_name not in df.columns:
        return []

    values = df[column_name].dropna().astype("string").str.strip()
    values = values[values != ""]
    return sorted(values.unique().tolist())


def get_configured_unique_values(
    dataframes: dict[str, pd.DataFrame],
    unique_config: dict,
) -> dict[str, dict[str, list[str]]]:
    """Return unique values only for the table/column pairs requested in config."""
    result: dict[str, dict[str, list[str]]] = {}

    for requested_table, table_config in unique_config.items():
        resolved_table = resolve_table_name(requested_table, dataframes)

        if resolved_table is None:
            result[requested_table] = {
                "_warning": [f"Table '{requested_table}' not found in source database."]
            }
            continue

        columns = table_config.get("columns", [])
        df = dataframes[resolved_table]

        result[resolved_table] = {
            column_name: get_unique_values(df, column_name)
            for column_name in columns
        }

    return result


In [44]:
unique_config = config["profiling"]["unique_values"]
unique_values = get_configured_unique_values(dataframes, unique_config)

unique_values_frame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "column_name": column_name,
            "unique_values": values,
            "unique_count": len(values),
        }
        for table_name, columns in unique_values.items()
        for column_name, values in columns.items()
        if column_name != "_warning"
    ]
).sort_values(["table_name", "column_name"], ignore_index=True)

display(unique_values_frame)
unique_values


,table_name,column_name,unique_values,unique_count
0,employees,role,"[Barista, Cashier, Manager, Waiter, Waitress, ...",8
1,inventory_tracking,reason,"[Damaged, ERROR, Expired, Restock]",4
2,orders,payment_method,"[Cash, Credit Card, Debit Card, ERROR]",4
3,products,category,"[Coffee, Pastry, Salad, Sandwich, Smoothie, Tea]",6


{'employees': {'role': ['Barista',
   'Cashier',
   'Manager',
   'Waiter',
   'Waitress',
   'me',
   'third',
   'today']},
 'orders': {'payment_method': ['Cash', 'Credit Card', 'Debit Card', 'ERROR']},
 'products': {'category': ['Coffee',
   'Pastry',
   'Salad',
   'Sandwich',
   'Smoothie',
   'Tea']},
 'inventory_tracking': {'reason': ['Damaged', 'ERROR', 'Expired', 'Restock']}}

## 1.6 Create Data Profiling Report


In [45]:
def build_data_profiling_report(
    person_in_charge: str,
    dataframes: dict[str, pd.DataFrame],
    unique_values_result: dict[str, dict[str, list[str]]],
) -> dict:
    """Build the profiling report in dictionary form."""
    report = {
        "person_in_charge": person_in_charge,
        "date_profiling": datetime.now().isoformat(timespec="seconds"),
        "result": {},
    }

    for table_name, df in dataframes.items():
        report["result"][table_name] = {
            "shape": get_table_shape(df),
            "data_types": get_column_data_types(df),
            "unique_values": unique_values_result.get(table_name, {}),
        }

    return report


def save_json_report(report: dict, output_path: Path) -> Path:
    """Save a dictionary report to JSON."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=2)

    return output_path


In [46]:
profiling_report = build_data_profiling_report(
    person_in_charge=config["project"]["person_in_charge"],
    dataframes=dataframes,
    unique_values_result=unique_values,
)

profiling_report_path = PROJECT_ROOT / config["paths"]["profiling_report_path"]
save_json_report(profiling_report, profiling_report_path)

print("Profiling report saved to:", profiling_report_path)
profiling_report["result"]["employees"]


Profiling report saved to: /Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1/outputs/data_profiling_report.json


{'shape': [103, 7],
 'data_types': {'employee_id': 'int64',
  'first_name': 'str',
  'last_name': 'str',
  'hire_date': 'object',
  'role': 'str',
  'email': 'str',
  'created_at': 'datetime64[us]'},
 'unique_values': {'role': ['Barista',
   'Cashier',
   'Manager',
   'Waiter',
   'Waitress',
   'me',
   'third',
   'today']}}

# Data Quality


## 2.1 Check Missing Values


In [47]:
PLACEHOLDER_TOKENS = {"", "na", "n/a", "null", "none", "nan"}


def check_missing_values(df: pd.DataFrame) -> dict[str, int]:
    """Count null values for every column in one DataFrame."""
    return df.isna().sum().astype(int).to_dict()


def check_missing_values_all_tables(dataframes: dict[str, pd.DataFrame]) -> dict[str, dict[str, int]]:
    """Count null values for every column in all tables."""
    return {
        table_name: check_missing_values(df)
        for table_name, df in dataframes.items()
    }


def check_blank_like_values(df: pd.DataFrame) -> dict[str, int]:
    """Supplemental completeness check for blank strings and placeholder tokens."""
    result: dict[str, int] = {}

    for column_name in df.columns:
        series = df[column_name]

        if is_object_dtype(series) or is_string_dtype(series):
            normalized = series.astype("string").str.strip().str.lower()
            result[column_name] = int(normalized.isin(PLACEHOLDER_TOKENS).sum())
        else:
            result[column_name] = 0

    return result


def check_blank_like_values_all_tables(
    dataframes: dict[str, pd.DataFrame]
) -> dict[str, dict[str, int]]:
    """Run the blank-like completeness check across all tables."""
    return {
        table_name: check_blank_like_values(df)
        for table_name, df in dataframes.items()
    }


def build_count_frame(result: dict[str, dict[str, int]], value_name: str) -> pd.DataFrame:
    rows = [
        {
            "table_name": table_name,
            "column_name": column_name,
            value_name: count,
        }
        for table_name, columns in result.items()
        for column_name, count in columns.items()
    ]
    return pd.DataFrame(rows).sort_values(
        ["table_name", value_name, "column_name"],
        ascending=[True, False, True],
        ignore_index=True,
    )


In [48]:
missing_values_result = check_missing_values_all_tables(dataframes)
blank_like_values_result = check_blank_like_values_all_tables(dataframes)

missing_values_frame = build_count_frame(missing_values_result, "missing_count")
blank_like_values_frame = build_count_frame(blank_like_values_result, "blank_like_count")

display(missing_values_frame[missing_values_frame["missing_count"] > 0])
display(blank_like_values_frame[blank_like_values_frame["blank_like_count"] > 0])


,table_name,column_name,missing_count
0,customers,phone,4
27,orders,customer_id,250


,table_name,column_name,blank_like_count


## 2.2 Date Validation


In [49]:
def validate_date_column(series: pd.Series, date_format: str) -> dict[str, object]:
    """
    Validate a date or timestamp column.

    If the column is already loaded as a pandas datetime dtype from PostgreSQL,
    the validation is treated as dtype-based because the raw text format is no
    longer available after extraction.
    """
    non_null_values = series.dropna()

    if non_null_values.empty:
        return {
            "status": "valid",
            "total_checked": 0,
            "invalid_count": 0,
            "invalid_examples": [],
            "expected_format": date_format,
            "validation_mode": "empty",
        }

    if is_datetime64_any_dtype(series):
        parsed_dates = pd.to_datetime(non_null_values, errors="coerce")
        invalid_values = non_null_values[parsed_dates.isna()]

        return {
            "status": "valid" if invalid_values.empty else "invalid",
            "total_checked": int(non_null_values.shape[0]),
            "invalid_count": int(invalid_values.shape[0]),
            "invalid_examples": invalid_values.astype("string").head(5).tolist(),
            "expected_format": date_format,
            "validation_mode": "dtype_based",
        }

    normalized_values = non_null_values.astype("string").str.strip()
    parsed_dates = pd.to_datetime(normalized_values, format=date_format, errors="coerce")
    invalid_values = normalized_values[parsed_dates.isna()]

    return {
        "status": "valid" if invalid_values.empty else "invalid",
        "total_checked": int(normalized_values.shape[0]),
        "invalid_count": int(invalid_values.shape[0]),
        "invalid_examples": invalid_values.head(5).tolist(),
        "expected_format": date_format,
        "validation_mode": "format_based",
    }


def validate_dates_all_tables(
    dataframes: dict[str, pd.DataFrame],
    date_config: dict,
) -> dict[str, dict[str, dict[str, object]]]:
    """Validate configured date columns for all tables."""
    result: dict[str, dict[str, dict[str, object]]] = {}

    for table_name, columns_config in date_config.items():
        if table_name not in dataframes:
            result[table_name] = {
                "_warning": {
                    "status": "invalid",
                    "total_checked": 0,
                    "invalid_count": 0,
                    "invalid_examples": [f"Table '{table_name}' not found."],
                    "expected_format": "n/a",
                    "validation_mode": "warning",
                }
            }
            continue

        df = dataframes[table_name]
        result[table_name] = {}

        for column_name, expected_format in columns_config.items():
            if column_name not in df.columns:
                result[table_name][column_name] = {
                    "status": "invalid",
                    "total_checked": 0,
                    "invalid_count": 0,
                    "invalid_examples": [f"Column '{column_name}' not found."],
                    "expected_format": expected_format,
                    "validation_mode": "warning",
                }
                continue

            result[table_name][column_name] = validate_date_column(
                df[column_name],
                expected_format,
            )

    return result


def flatten_validation_results(result: dict[str, dict[str, dict[str, object]]]) -> pd.DataFrame:
    rows = [
        {
            "table_name": table_name,
            "column_name": column_name,
            **details,
        }
        for table_name, columns in result.items()
        for column_name, details in columns.items()
        if isinstance(details, dict)
    ]
    return pd.DataFrame(rows).sort_values(
        ["table_name", "column_name"],
        ignore_index=True,
    )


In [50]:
date_validation_config = config["data_quality"]["date_validation"]
date_validation_result = validate_dates_all_tables(dataframes, date_validation_config)

date_validation_frame = flatten_validation_results(date_validation_result)
display(date_validation_frame)


,table_name,column_name,status,total_checked,invalid_count,invalid_examples,expected_format,validation_mode
0,employees,hire_date,valid,103,0,[],%Y-%m-%d,format_based
1,inventory_tracking,change_date,valid,162,0,[],%Y-%m-%d,dtype_based
2,orders,order_date,valid,1010,0,[],%Y-%m-%d %H:%M:%S,dtype_based


## 2.3 Numeric Validation


In [51]:
def prepare_numeric_series(series: pd.Series) -> pd.Series:
    """
    Prepare numeric-like strings for supplemental checks.

    This keeps the strict numeric validation honest while still allowing
    later checks such as negative-value detection on values like `$-15.00`.
    """
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.replace(
        {
            "": pd.NA,
            "NA": pd.NA,
            "N/A": pd.NA,
            "NULL": pd.NA,
            "None": pd.NA,
            "nan": pd.NA,
        }
    )
    cleaned = cleaned.str.replace(",", "", regex=False)
    cleaned = cleaned.str.replace("$", "", regex=False)
    cleaned = cleaned.str.replace(r"^\((.*)\)$", r"-\1", regex=True)
    return cleaned


def validate_numeric_column(series: pd.Series) -> dict[str, object]:
    """
    Strict numeric validation on the raw extracted values.

    If a value only becomes numeric after cleaning (for example `$47.00`),
    it is still marked invalid in the raw dataset but reported as recoverable.
    """
    non_null_values = series.dropna().astype("string").str.strip()
    non_null_values = non_null_values[non_null_values != ""]

    if non_null_values.empty:
        return {
            "status": "valid",
            "total_checked": 0,
            "invalid_count": 0,
            "invalid_examples": [],
            "recoverable_after_cleaning_count": 0,
            "recoverable_examples": [],
            "unrecoverable_count": 0,
            "unrecoverable_examples": [],
        }

    strict_numeric = pd.to_numeric(non_null_values, errors="coerce")
    cleaned_values = prepare_numeric_series(non_null_values)
    cleaned_numeric = pd.to_numeric(cleaned_values, errors="coerce")

    invalid_mask = strict_numeric.isna()
    recoverable_mask = invalid_mask & cleaned_numeric.notna()
    unrecoverable_mask = invalid_mask & cleaned_numeric.isna()

    return {
        "status": "valid" if int(invalid_mask.sum()) == 0 else "invalid",
        "total_checked": int(non_null_values.shape[0]),
        "invalid_count": int(invalid_mask.sum()),
        "invalid_examples": non_null_values[invalid_mask].head(5).tolist(),
        "recoverable_after_cleaning_count": int(recoverable_mask.sum()),
        "recoverable_examples": non_null_values[recoverable_mask].head(5).tolist(),
        "unrecoverable_count": int(unrecoverable_mask.sum()),
        "unrecoverable_examples": non_null_values[unrecoverable_mask].head(5).tolist(),
    }


def validate_numeric_all_tables(
    dataframes: dict[str, pd.DataFrame],
    numeric_config: dict,
) -> dict[str, dict[str, dict[str, object]]]:
    """Validate configured numeric columns for all tables."""
    result: dict[str, dict[str, dict[str, object]]] = {}

    for table_name, table_config in numeric_config.items():
        if table_name not in dataframes:
            result[table_name] = {
                "_warning": {
                    "status": "invalid",
                    "total_checked": 0,
                    "invalid_count": 0,
                    "invalid_examples": [f"Table '{table_name}' not found."],
                    "recoverable_after_cleaning_count": 0,
                    "recoverable_examples": [],
                    "unrecoverable_count": 0,
                    "unrecoverable_examples": [],
                }
            }
            continue

        df = dataframes[table_name]
        result[table_name] = {}

        for column_name in table_config.get("columns", []):
            if column_name not in df.columns:
                result[table_name][column_name] = {
                    "status": "invalid",
                    "total_checked": 0,
                    "invalid_count": 0,
                    "invalid_examples": [f"Column '{column_name}' not found."],
                    "recoverable_after_cleaning_count": 0,
                    "recoverable_examples": [],
                    "unrecoverable_count": 0,
                    "unrecoverable_examples": [],
                }
                continue

            result[table_name][column_name] = validate_numeric_column(df[column_name])

    return result


In [52]:
numeric_validation_config = config["data_quality"]["numeric_validation"]
numeric_validation_result = validate_numeric_all_tables(dataframes, numeric_validation_config)

numeric_validation_frame = flatten_validation_results(numeric_validation_result)
display(numeric_validation_frame)


,table_name,column_name,status,total_checked,invalid_count,invalid_examples,recoverable_after_cleaning_count,recoverable_examples,unrecoverable_count,unrecoverable_examples
0,customers,loyalty_points,valid,204,0,[],0,[],0,[]
1,inventory_tracking,quantity_change,valid,162,0,[],0,[],0,[]
2,order_details,quantity,valid,3022,0,[],0,[],0,[]
3,order_details,subtotal,valid,3022,0,[],0,[],0,[]
4,order_details,unit_price,valid,3022,0,[],0,[],0,[]
5,orders,total_amount,valid,1010,0,[],0,[],0,[]
6,products,cost_price,invalid,54,54,"[$43.00, $43.00, $14.00, $20.00, $40.00]",54,"[$43.00, $43.00, $14.00, $20.00, $40.00]",0,[]
7,products,unit_price,invalid,54,54,"[$47.00, $47.00, $17.00, $21.00, $43.00]",54,"[$47.00, $47.00, $17.00, $21.00, $43.00]",0,[]


## 2.4 Negative Value Validation


In [53]:
def validate_negative_column(series: pd.Series) -> dict[str, object]:
    """
    Validate that no negative values exist.

    This uses cleaned numeric-like values so that formatted strings such as
    `$-15.00` are still inspected for sign.
    """
    prepared_values = prepare_numeric_series(series)
    numeric_values = pd.to_numeric(prepared_values, errors="coerce")
    checked_values = numeric_values.dropna()

    if checked_values.empty:
        return {
            "status": "valid",
            "total_checked": 0,
            "negative_count": 0,
            "negative_examples": [],
            "unparsed_count": int(prepared_values.notna().sum()),
        }

    negative_values = checked_values[checked_values < 0]

    return {
        "status": "valid" if negative_values.empty else "invalid",
        "total_checked": int(checked_values.shape[0]),
        "negative_count": int(negative_values.shape[0]),
        "negative_examples": negative_values.head(5).tolist(),
        "unparsed_count": int(prepared_values.notna().sum() - checked_values.shape[0]),
    }


def validate_negative_all_tables(
    dataframes: dict[str, pd.DataFrame],
    negative_config: dict,
) -> dict[str, dict[str, dict[str, object]]]:
    """Validate negative values for all configured columns."""
    result: dict[str, dict[str, dict[str, object]]] = {}

    for table_name, table_config in negative_config.items():
        if table_name not in dataframes:
            result[table_name] = {
                "_warning": {
                    "status": "invalid",
                    "total_checked": 0,
                    "negative_count": 0,
                    "negative_examples": [f"Table '{table_name}' not found."],
                    "unparsed_count": 0,
                }
            }
            continue

        df = dataframes[table_name]
        allow_negative = table_config.get("allow_negative", False)
        result[table_name] = {}

        for column_name in table_config.get("columns", []):
            if column_name not in df.columns:
                result[table_name][column_name] = {
                    "status": "invalid",
                    "total_checked": 0,
                    "negative_count": 0,
                    "negative_examples": [f"Column '{column_name}' not found."],
                    "unparsed_count": 0,
                }
                continue

            validation = validate_negative_column(df[column_name])

            if allow_negative:
                validation = {
                    **validation,
                    "status": "valid",
                }

            result[table_name][column_name] = validation

    return result


In [54]:
negative_validation_config = config["data_quality"]["negative_validation"]
negative_validation_result = validate_negative_all_tables(dataframes, negative_validation_config)

negative_validation_frame = flatten_validation_results(negative_validation_result)
display(negative_validation_frame)


,table_name,column_name,status,total_checked,negative_count,negative_examples,unparsed_count
0,customers,loyalty_points,invalid,204,4,"[-710, -525, -304, -993]",0
1,inventory_tracking,quantity_change,valid,162,0,[],0
2,order_details,quantity,valid,3022,0,[],0
3,order_details,subtotal,valid,3022,0,[],0
4,order_details,unit_price,valid,3022,0,[],0
5,orders,total_amount,valid,1010,0,[],0
6,products,cost_price,invalid,54,3,"[-3.0, -3.0, -2.0]",0
7,products,unit_price,invalid,54,2,"[-15.0, -11.0]",0


## 2.5 Build Data Quality Report


In [55]:
def build_status_map(validation_result: dict[str, dict[str, object]]) -> dict[str, str]:
    return {
        column_name: details.get("status", "invalid")
        for column_name, details in validation_result.items()
        if isinstance(details, dict)
    }


def build_data_quality_report(
    person_in_charge: str,
    dataframes: dict[str, pd.DataFrame],
    missing_values: dict[str, dict[str, int]],
    blank_like_values: dict[str, dict[str, int]],
    date_validity: dict[str, dict[str, dict[str, object]]],
    numeric_validity: dict[str, dict[str, dict[str, object]]],
    negative_validity: dict[str, dict[str, dict[str, object]]],
) -> dict:
    """Build the data quality report while keeping detailed diagnostics."""
    report = {
        "person_in_charge": person_in_charge,
        "date_quality_check": datetime.now().isoformat(timespec="seconds"),
        "result": {},
    }

    for table_name in dataframes:
        report["result"][table_name] = {
            "missing_values": missing_values.get(table_name, {}),
            "blank_like_values": blank_like_values.get(table_name, {}),
            "date_validity": build_status_map(date_validity.get(table_name, {})),
            "numeric_validity": build_status_map(numeric_validity.get(table_name, {})),
            "negative_validity": build_status_map(negative_validity.get(table_name, {})),
            "date_validity_details": date_validity.get(table_name, {}),
            "numeric_validity_details": numeric_validity.get(table_name, {}),
            "negative_validity_details": negative_validity.get(table_name, {}),
        }

    return report


In [56]:
quality_report = build_data_quality_report(
    person_in_charge=config["project"]["person_in_charge"],
    dataframes=dataframes,
    missing_values=missing_values_result,
    blank_like_values=blank_like_values_result,
    date_validity=date_validation_result,
    numeric_validity=numeric_validation_result,
    negative_validity=negative_validation_result,
)

quality_report_path = PROJECT_ROOT / config["paths"]["quality_report_path"]
save_json_report(quality_report, quality_report_path)

print("Quality report saved to:", quality_report_path)
quality_report["result"]["products"]


Quality report saved to: /Users/circlesnclown/CodeSavesMe/pacmann/data_pipeline_with_python_and_spark/mentoring_1/outputs/data_quality_report.json


{'missing_values': {'product_id': 0,
  'product_name': 0,
  'category': 0,
  'unit_price': 0,
  'cost_price': 0,
  'in_stock': 0,
  'created_at': 0},
 'blank_like_values': {'product_id': 0,
  'product_name': 0,
  'category': 0,
  'unit_price': 0,
  'cost_price': 0,
  'in_stock': 0,
  'created_at': 0},
 'date_validity': {},
 'numeric_validity': {'unit_price': 'invalid', 'cost_price': 'invalid'},
 'negative_validity': {'unit_price': 'invalid', 'cost_price': 'invalid'},
 'date_validity_details': {},
 'numeric_validity_details': {'unit_price': {'status': 'invalid',
   'total_checked': 54,
   'invalid_count': 54,
   'invalid_examples': ['$47.00', '$47.00', '$17.00', '$21.00', '$43.00'],
   'recoverable_after_cleaning_count': 54,
   'recoverable_examples': ['$47.00', '$47.00', '$17.00', '$21.00', '$43.00'],
   'unrecoverable_count': 0,
   'unrecoverable_examples': []},
  'cost_price': {'status': 'invalid',
   'total_checked': 54,
   'invalid_count': 54,
   'invalid_examples': ['$43.00', '$43

In [57]:
output_report_paths = {
    "profiling_report": PROJECT_ROOT / config["paths"]["profiling_report_path"],
    "quality_report": PROJECT_ROOT / config["paths"]["quality_report_path"],
}

loaded_output_reports = {}

for report_name, report_path in output_report_paths.items():
    with open(report_path, "r", encoding="utf-8") as file:
        loaded_output_reports[report_name] = json.load(file)

output_report_summary = pd.DataFrame(
    [
        {
            "report_name": report_name,
            "file_path": str(report_path.relative_to(PROJECT_ROOT)),
            "generated_at": report.get("date_profiling", report.get("date_quality_check")),
            "person_in_charge": report.get("person_in_charge"),
            "table_count": len(report.get("result", {})),
        }
        for report_name, report_path in output_report_paths.items()
        for report in [loaded_output_reports[report_name]]
    ]
).sort_values("report_name", ignore_index=True)

profiling_output_frame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "rows": details["shape"][0],
            "columns": details["shape"][1],
            "data_type_count": len(details.get("data_types", {})),
            "unique_columns_checked": len(details.get("unique_values", {})),
        }
        for table_name, details in loaded_output_reports["profiling_report"]["result"].items()
    ]
).sort_values("table_name", ignore_index=True)

quality_output_frame = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "total_missing_values": sum(details.get("missing_values", {}).values()),
            "columns_with_missing_values": sum(
                1 for count in details.get("missing_values", {}).values() if count > 0
            ),
            "total_blank_like_values": sum(details.get("blank_like_values", {}).values()),
            "invalid_date_columns": ", ".join(
                sorted(
                    column_name
                    for column_name, status in details.get("date_validity", {}).items()
                    if status == "invalid"
                )
            ) or "-",
            "invalid_numeric_columns": ", ".join(
                sorted(
                    column_name
                    for column_name, status in details.get("numeric_validity", {}).items()
                    if status == "invalid"
                )
            ) or "-",
            "invalid_negative_columns": ", ".join(
                sorted(
                    column_name
                    for column_name, status in details.get("negative_validity", {}).items()
                    if status == "invalid"
                )
            ) or "-",
        }
        for table_name, details in loaded_output_reports["quality_report"]["result"].items()
    ]
).sort_values("table_name", ignore_index=True)

print("Loaded JSON reports from mentoring_1/outputs")
display(output_report_summary)

print("Profiling report summary")
display(profiling_output_frame)

print("Quality report summary")
display(quality_output_frame)


Loaded JSON reports from mentoring_1/outputs


,report_name,file_path,generated_at,person_in_charge,table_count
0,profiling_report,outputs/data_profiling_report.json,2026-05-05T01:11:40,Akbar,6
1,quality_report,outputs/data_quality_report.json,2026-05-05T01:11:41,Akbar,6


Profiling report summary


,table_name,rows,columns,data_type_count,unique_columns_checked
0,customers,204,7,7,0
1,employees,103,7,7,1
2,inventory_tracking,162,6,6,1
3,order_details,3022,7,7,0
4,orders,1010,8,8,1
5,products,54,7,7,1


Quality report summary


,table_name,total_missing_values,columns_with_missing_values,total_blank_like_values,invalid_date_columns,invalid_numeric_columns,invalid_negative_columns
0,customers,4,1,0,-,-,loyalty_points
1,employees,0,0,0,-,-,-
2,inventory_tracking,0,0,0,-,-,-
3,order_details,0,0,0,-,-,-
4,orders,250,1,0,-,-,-
5,products,0,0,0,-,"cost_price, unit_price","cost_price, unit_price"


# Output Result

# Recommendations


In [58]:
EXPECTED_CATEGORICAL_VALUES = {
    "employees": {
        "role": {"Barista", "Cashier", "Manager", "Waiter", "Waitress"},
    },
    "orders": {
        "payment_method": {"Cash", "Credit Card", "Debit Card"},
    },
    "inventory_tracking": {
        "reason": {"Restock", "Damaged", "Expired"},
    },
}


def find_unexpected_categorical_values(
    dataframes: dict[str, pd.DataFrame],
    expected_values: dict[str, dict[str, set[str]]],
) -> pd.DataFrame:
    rows = []

    for table_name, columns in expected_values.items():
        if table_name not in dataframes:
            continue

        df = dataframes[table_name]

        for column_name, allowed_values in columns.items():
            if column_name not in df.columns:
                continue

            normalized = df[column_name].dropna().astype("string").str.strip()
            counts = normalized.value_counts()

            for value, count in counts.items():
                if value not in allowed_values:
                    rows.append(
                        {
                            "table_name": table_name,
                            "column_name": column_name,
                            "unexpected_value": value,
                            "occurrences": int(count),
                        }
                    )

    return pd.DataFrame(rows).sort_values(
        ["table_name", "column_name", "occurrences", "unexpected_value"],
        ascending=[True, True, False, True],
        ignore_index=True,
    )


def check_product_price_business_rules(products_df: pd.DataFrame) -> dict[str, int]:
    unit_prices = pd.to_numeric(
        prepare_numeric_series(products_df["unit_price"]),
        errors="coerce",
    )
    cost_prices = pd.to_numeric(
        prepare_numeric_series(products_df["cost_price"]),
        errors="coerce",
    )

    return {
        "negative_unit_price_count": int((unit_prices < 0).sum()),
        "negative_cost_price_count": int((cost_prices < 0).sum()),
        "cost_greater_than_unit_count": int((cost_prices > unit_prices).fillna(False).sum()),
    }


def check_inventory_direction_rules(inventory_df: pd.DataFrame) -> dict[str, int]:
    reasons = inventory_df["reason"].astype("string").str.strip()
    quantity_change = pd.to_numeric(
        prepare_numeric_series(inventory_df["quantity_change"]),
        errors="coerce",
    )

    return {
        "damaged_or_expired_positive_count": int(
            ((reasons.isin(["Damaged", "Expired"])) & (quantity_change > 0)).sum()
        ),
        "restock_negative_count": int(
            ((reasons == "Restock") & (quantity_change < 0)).sum()
        ),
    }


def generate_recommendations(
    dataframes: dict[str, pd.DataFrame],
    missing_values: dict[str, dict[str, int]],
    numeric_validity: dict[str, dict[str, dict[str, object]]],
    negative_validity: dict[str, dict[str, dict[str, object]]],
) -> pd.DataFrame:
    recommendations: list[dict[str, str]] = []

    categorical_issues = find_unexpected_categorical_values(
        dataframes,
        EXPECTED_CATEGORICAL_VALUES,
    )
    product_price_issues = check_product_price_business_rules(dataframes["products"])
    inventory_direction_issues = check_inventory_direction_rules(dataframes["inventory_tracking"])

    order_customer_missing = missing_values["orders"]["customer_id"]
    if order_customer_missing > 0:
        recommendations.append(
            {
                "priority": "High",
                "finding": (
                    f"`orders.customer_id` has {order_customer_missing} missing values out of "
                    f"{len(dataframes['orders'])} orders."
                ),
                "recommendation": (
                    "Confirm whether these records are guest purchases. If yes, document the rule and "
                    "add an `is_guest_order` flag instead of filling fake customer IDs."
                ),
            }
        )

    products_numeric = numeric_validity["products"]
    products_negative = negative_validity["products"]
    if any(details.get("status") == "invalid" for details in products_numeric.values()):
        recommendations.append(
            {
                "priority": "High",
                "finding": (
                    "`products.unit_price` and `products.cost_price` are stored as strings and fail strict "
                    "numeric validation. "
                    f"Negative unit prices: {product_price_issues['negative_unit_price_count']}. "
                    f"Negative cost prices: {product_price_issues['negative_cost_price_count']}. "
                    f"Rows with `cost_price > unit_price`: {product_price_issues['cost_greater_than_unit_count']}."
                ),
                "recommendation": (
                    "Strip currency symbols, cast both columns to decimal, reject negative prices, and "
                    "review rows where the cost is higher than the selling price."
                ),
            }
        )

    if not categorical_issues.empty:
        payment_issues = categorical_issues[
            (categorical_issues["table_name"] == "orders")
            & (categorical_issues["column_name"] == "payment_method")
        ]
        role_issues = categorical_issues[
            (categorical_issues["table_name"] == "employees")
            & (categorical_issues["column_name"] == "role")
        ]
        reason_issues = categorical_issues[
            (categorical_issues["table_name"] == "inventory_tracking")
            & (categorical_issues["column_name"] == "reason")
        ]

        if not payment_issues.empty:
            recommendations.append(
                {
                    "priority": "High",
                    "finding": (
                        f"`orders.payment_method` contains unexpected values: "
                        f"{payment_issues[['unexpected_value', 'occurrences']].to_dict('records')}"
                    ),
                    "recommendation": (
                        "Replace invalid payment methods from the source when possible. If the real method "
                        "cannot be recovered, map the records to a controlled fallback such as `unknown`."
                    ),
                }
            )

        if not role_issues.empty:
            recommendations.append(
                {
                    "priority": "Medium",
                    "finding": (
                        f"`employees.role` contains unexpected values: "
                        f"{role_issues[['unexpected_value', 'occurrences']].to_dict('records')}"
                    ),
                    "recommendation": (
                        "Standardize roles to the approved set and add a validation rule or lookup table "
                        "to prevent free-text roles in future loads."
                    ),
                }
            )

        if not reason_issues.empty:
            recommendations.append(
                {
                    "priority": "Medium",
                    "finding": (
                        f"`inventory_tracking.reason` contains unexpected values: "
                        f"{reason_issues[['unexpected_value', 'occurrences']].to_dict('records')}. "
                        f"Rows where `Damaged` or `Expired` still increase stock: "
                        f"{inventory_direction_issues['damaged_or_expired_positive_count']}."
                    ),
                    "recommendation": (
                        "Standardize inventory movement reasons and define the sign rule for "
                        "`quantity_change`, especially for shrinkage events such as damage and expiry."
                    ),
                }
            )

    customer_phone_missing = missing_values["customers"]["phone"]
    customer_loyalty_negative = negative_validity["customers"]["loyalty_points"]["negative_count"]
    if customer_phone_missing > 0 or customer_loyalty_negative > 0:
        recommendations.append(
            {
                "priority": "Medium",
                "finding": (
                    f"`customers.phone` has {customer_phone_missing} missing values and "
                    f"`customers.loyalty_points` has {customer_loyalty_negative} negative values."
                ),
                "recommendation": (
                    "Backfill phone numbers if a trusted source exists. Review negative loyalty balances "
                    "and decide whether they are errors or valid adjustment outcomes that need a separate transaction model."
                ),
            }
        )

    recommendations.append(
        {
            "priority": "Medium",
            "finding": "The dataset depends on business assumptions that are not yet enforced in the schema.",
            "recommendation": (
                "Document quality rules and add database constraints or ingestion checks for allowed categories, "
                "numeric typing, non-negative prices, and expected inventory movement behavior."
            ),
        }
    )

    return pd.DataFrame(recommendations)


categorical_issue_frame = find_unexpected_categorical_values(
    dataframes,
    EXPECTED_CATEGORICAL_VALUES,
)
recommendations_frame = generate_recommendations(
    dataframes=dataframes,
    missing_values=missing_values_result,
    numeric_validity=numeric_validation_result,
    negative_validity=negative_validation_result,
)

display(categorical_issue_frame)
display(recommendations_frame)

for idx, row in recommendations_frame.iterrows():
    print(f"{idx + 1}. [{row['priority']}] {row['finding']}")
    print(f"   Recommendation: {row['recommendation']}")


,table_name,column_name,unexpected_value,occurrences
0,employees,role,me,1
1,employees,role,third,1
2,employees,role,today,1
3,inventory_tracking,reason,ERROR,10
4,orders,payment_method,ERROR,282


,priority,finding,recommendation
0,High,`orders.customer_id` has 250 missing values ou...,Confirm whether these records are guest purcha...
1,High,`products.unit_price` and `products.cost_price...,"Strip currency symbols, cast both columns to d..."
2,High,`orders.payment_method` contains unexpected va...,Replace invalid payment methods from the sourc...
3,Medium,`employees.role` contains unexpected values: [...,Standardize roles to the approved set and add ...
4,Medium,`inventory_tracking.reason` contains unexpecte...,Standardize inventory movement reasons and def...
5,Medium,`customers.phone` has 4 missing values and `cu...,Backfill phone numbers if a trusted source exi...
6,Medium,The dataset depends on business assumptions th...,Document quality rules and add database constr...


1. [High] `orders.customer_id` has 250 missing values out of 1010 orders.
   Recommendation: Confirm whether these records are guest purchases. If yes, document the rule and add an `is_guest_order` flag instead of filling fake customer IDs.
2. [High] `products.unit_price` and `products.cost_price` are stored as strings and fail strict numeric validation. Negative unit prices: 2. Negative cost prices: 3. Rows with `cost_price > unit_price`: 4.
   Recommendation: Strip currency symbols, cast both columns to decimal, reject negative prices, and review rows where the cost is higher than the selling price.
3. [High] `orders.payment_method` contains unexpected values: [{'unexpected_value': 'ERROR', 'occurrences': 282}]
   Recommendation: Replace invalid payment methods from the source when possible. If the real method cannot be recovered, map the records to a controlled fallback such as `unknown`.
4. [Medium] `employees.role` contains unexpected values: [{'unexpected_value': 'me', 'occurrenc